# Calor Urbano y Clima en Santiago — Análisis de Exposome

**Objetivo:** construir un indicador geoespacial robusto de **exposición a calor y clima por comuna** en la Región Metropolitana de Santiago, usando datos abiertos y reproducibles.

Este notebook agrega la capa que faltaba en la serie del exposome: **variables climáticas**. Comparte la misma unidad geográfica (`name`, comuna) que los notebooks de salud, calidad del aire, áreas verdes y nivel socioeconómico, para que todos los indicadores puedan cruzarse con datos clínicos o censales.

**Por qué calor urbano:** la exposición crónica a altas temperaturas, olas de calor, noches cálidas y estrés térmico afecta sueño, riesgo cardiovascular, actividad física, salud mental y vulnerabilidad en personas mayores. En estudios de envejecimiento cerebral, además, el calor urbano es un determinante ambiental plausible y cada vez más medible.

**Decisión metodológica clave:** el indicador principal usa un **punto representativo dentro de cada comuna** como proxy residencial. Esto evita que comunas grandes con cordillera o zonas rurales no habitadas queden artificialmente dominadas por alta montaña. La grilla regular se conserva como diagnóstico espacial y como fallback.

**Fuentes de datos:**
- [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api), con reanálisis histórico sin API key.
- Límites comunales desde salidas locales `*_exposome_rm_santiago.geojson` si existen; si no, desde OpenStreetMap vía `osmnx`.

**Indicadores generados (año 2024):**
- `tmean_annual_c` — temperatura media anual.
- `tmax_mean_annual_c` — temperatura máxima diaria promedio anual.
- `summer_tmax_mean_c` — temperatura máxima diaria promedio en meses cálidos (enero, febrero, diciembre).
- `tmax_p95_c` y `tmax_abs_c` — percentil 95 y máximo anual de temperatura máxima diaria.
- `hot_days_30c`, `hot_days_35c` — días con temperatura máxima ≥30 °C y ≥35 °C.
- `apparent_hot_days_35c` — días con temperatura aparente máxima ≥35 °C.
- `tropical_nights_20c` — noches con temperatura mínima ≥20 °C.
- `precip_annual_mm` — precipitación anual acumulada, como contexto climático.
- `heat_exposure_index` — índice compuesto de calor relativo dentro de la RM; mayor = peor exposición térmica.

## Plan

1. Reutilizar las 52 comunas oficiales de la RM desde salidas locales si están disponibles; si no, descargarlas desde OSM.
2. Construir una grilla climática regular de ~0.1° y agregar centroides comunales para evitar comunas pequeñas sin muestra.
3. Descargar series diarias de Open-Meteo para 2024 con caché reanudable y backoff ante rate limits.
4. Resumir exposición térmica por punto y agregar a comuna mediante `spatial join` con fallback al punto más cercano.
5. Exportar CSV/GeoJSON, mapas de publicación, mapa interactivo y ranking.
6. Integrar el nuevo factor con NSE, áreas verdes, calidad del aire y salud para mostrar acumulación de vulnerabilidades.

In [ ]:
from __future__ import annotations

import json
import os
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

# Keep Matplotlib cache inside the project when the home directory is not writable.
Path("cache/matplotlib").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(Path("cache/matplotlib").resolve()))

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import Point

try:
    import osmnx as ox
except ImportError:
    ox = None

try:
    import folium
except ImportError:
    folium = None

warnings.filterwarnings("ignore")


def version_or_missing(module):
    return getattr(module, "__version__", "missing") if module is not None else "missing"

print(f"python    : {'.'.join(map(str, __import__('sys').version_info[:3]))}")
print(f"pandas    : {pd.__version__}")
print(f"geopandas : {gpd.__version__}")
print(f"osmnx     : {version_or_missing(ox)}")
print(f"folium    : {version_or_missing(folium)}")
print(f"requests  : {requests.__version__}")

In [ ]:
REGION = "Región Metropolitana de Santiago, Chile"
YEAR = 2024
YEAR_START = f"{YEAR}-01-01"
YEAR_END = f"{YEAR}-12-31"
TIMEZONE = "America/Santiago"

# Open-Meteo Historical Weather API. Default model is best_match; for multi-decade
# comparability you can set MODEL = "era5_land" if your API call supports it.
OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
MODEL = None
DAILY_VARS = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "apparent_temperature_max",
    "precipitation_sum",
]

# Heat thresholds: deliberately transparent and easy to adapt for sensitivity checks.
HOT_DAY_C = 30.0
VERY_HOT_DAY_C = 35.0
APPARENT_HOT_DAY_C = 35.0
TROPICAL_NIGHT_C = 20.0
SUMMER_MONTHS = {1, 2, 12}  # warm calendar months in central Chile

# Spatial setup.
METRIC_CRS = "EPSG:32719"  # WGS84 / UTM zone 19S, suitable for Santiago areas/distances
WGS84 = "EPSG:4326"
GRID_STEP_DEG = 0.10        # ~11 km; matches ERA5-Land scale and keeps API calls light
CHUNK_SIZE = 35             # Open-Meteo accepts comma-separated coordinate batches
REQUEST_SLEEP_S = 2.0

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
CLIMATE_CACHE = CACHE_DIR / f"climate_heat_grid_daily_{YEAR}.csv"
CLIMATE_PARTIAL = CACHE_DIR / f"climate_heat_grid_daily_{YEAR}.partial.csv"

OUT_CSV = Path("climate_heat_exposome_rm_santiago.csv")
OUT_GEOJSON = Path("climate_heat_exposome_rm_santiago.geojson")
OUT_META = Path("climate_heat_exposome_rm_santiago_metadata.json")
OUT_MAP = Path("climate_heat_santiago_pub.png")
OUT_HTML = Path("climate_heat_santiago.html")
OUT_RANKING = Path("climate_heat_ranking_santiago.png")
OUT_INTEGRATION = Path("heat_vs_exposome_santiago.png")

print(f"Año climático: {YEAR_START} → {YEAR_END}")
print(f"Caché climática: {CLIMATE_CACHE}")

## 1. Límites comunales

Para que el notebook sea reproducible pero también rápido al revisar la postulación, primero intenta reutilizar una capa GeoJSON ya generada por los notebooks anteriores. Si no existe, descarga las comunas desde OpenStreetMap y aplica el mismo filtro usado en el resto del proyecto.

In [ ]:
def load_communes(region: str = REGION) -> gpd.GeoDataFrame:
    """Load the 52 RM communes from local exposome outputs, falling back to OSM."""
    candidates = [
        "socioeconomic_exposome_rm_santiago.geojson",
        "air_quality_exposome_rm_santiago.geojson",
        "green_exposome_rm_santiago.geojson",
        "healthcare_exposome_rm_santiago.geojson",
    ]
    for fp in candidates:
        p = Path(fp)
        if p.exists():
            gdf = gpd.read_file(p).to_crs(METRIC_CRS)
            keep_cols = [c for c in ["name", "area_km2", "geometry"] if c in gdf.columns]
            gdf = gdf[keep_cols].copy()
            if "area_km2" not in gdf.columns:
                gdf["area_km2"] = gdf.geometry.area / 1e6
            gdf = gdf.dropna(subset=["name"]).drop_duplicates("name")
            print(f"Comunas cargadas desde {fp}: {len(gdf)}")
            return gdf.reset_index(drop=True)

    if ox is None:
        raise ImportError(
            "No hay GeoJSON comunal local y `osmnx` no está instalado. "
            "Instala osmnx o ejecuta primero uno de los notebooks previos para generar *_exposome_rm_santiago.geojson."
        )

    print("No encontré GeoJSON local; descargando límites comunales desde OSM...")
    raw = ox.features_from_place(region, tags={"boundary": "administrative", "admin_level": "8"})
    gdf = raw[raw.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy().to_crs(METRIC_CRS)
    gdf = gdf[["name", "geometry"]].dropna(subset=["name"]).dissolve(by="name").reset_index()

    rm_geom = ox.geocode_to_gdf(region).to_crs(METRIC_CRS).geometry.iloc[0]
    non_comuna = ("Provincia", "Región", "Dirección", "Departamento", "Distrito")
    gdf = gdf[
        gdf.geometry.centroid.within(rm_geom)
        & ~gdf["name"].str.startswith(non_comuna)
    ].reset_index(drop=True)
    gdf["area_km2"] = gdf.geometry.area / 1e6
    print(f"Comunas descargadas desde OSM: {len(gdf)}")
    return gdf


gdf_communes = load_communes()
gdf_communes["area_km2"] = (gdf_communes.to_crs(METRIC_CRS).geometry.area / 1e6).round(2)

if len(gdf_communes) != 52:
    raise ValueError(f"Esperaba 52 comunas de la RM y obtuve {len(gdf_communes)}. Revisar filtro geográfico.")

print(f"Área total comunas RM: {gdf_communes['area_km2'].sum():,.1f} km²")
gdf_communes[["name", "area_km2"]].sort_values("name").head()

## 2. Grilla climática

La grilla combina puntos regulares dentro de la RM y centroides comunales. Los centroides evitan que comunas pequeñas queden sin valor cuando la grilla del reanálisis es más gruesa que el polígono comunal.

In [ ]:
def build_climate_points(communes: gpd.GeoDataFrame, step_deg: float = GRID_STEP_DEG) -> gpd.GeoDataFrame:
    communes_wgs = communes.to_crs(WGS84)
    region_union = communes_wgs.geometry.union_all().buffer(0)
    minx, miny, maxx, maxy = communes_wgs.total_bounds

    lons = np.arange(np.floor(minx / step_deg) * step_deg, maxx + step_deg, step_deg)
    lats = np.arange(np.floor(miny / step_deg) * step_deg, maxy + step_deg, step_deg)

    regular = []
    for lat in lats:
        for lon in lons:
            pt = Point(float(lon), float(lat))
            if pt.within(region_union):
                regular.append({
                    "source": "grid",
                    "commune_name": None,
                    "lat": round(float(lat), 4),
                    "lon": round(float(lon), 4),
                    "geometry": pt,
                })

    cent = communes_wgs.copy()
    # representative_point is guaranteed to fall inside each polygon, unlike centroid for odd shapes.
    cent["geometry"] = cent.geometry.representative_point()
    centroids = [
        {
            "source": "commune_point",
            "commune_name": row["name"],
            "lat": round(row.geometry.y, 4),
            "lon": round(row.geometry.x, 4),
            "geometry": row.geometry,
        }
        for _, row in cent.iterrows()
    ]

    pts = gpd.GeoDataFrame(regular + centroids, crs=WGS84)
    pts = pts.drop_duplicates(subset=["source", "commune_name", "lat", "lon"]).reset_index(drop=True)
    pts["location_id"] = np.arange(len(pts), dtype=int)
    return pts[["location_id", "source", "commune_name", "lat", "lon", "geometry"]]


gdf_points = build_climate_points(gdf_communes)
print(f"Puntos climáticos: {len(gdf_points)} ({gdf_points['source'].value_counts().to_dict()})")
gdf_points.head()

## 3. Descargar clima diario desde Open-Meteo

La descarga usa:

- **caché completa** para reejecutar sin red;
- **caché parcial** para reanudar si se corta la conexión;
- **batches** de coordenadas;
- **reintentos con backoff** para rate limits o errores transitorios;
- validación de columnas al final.

In [ ]:
def _clean_daily_value(values):
    return [np.nan if v is None else v for v in values]


def fetch_open_meteo_chunk(chunk: pd.DataFrame, retries: int = 6) -> list[dict]:
    params = {
        "latitude": ",".join(chunk["lat"].astype(str)),
        "longitude": ",".join(chunk["lon"].astype(str)),
        "daily": ",".join(DAILY_VARS),
        "start_date": YEAR_START,
        "end_date": YEAR_END,
        "timezone": TIMEZONE,
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "cell_selection": "land",
    }
    if MODEL:
        params["models"] = MODEL

    for attempt in range(retries):
        try:
            r = requests.get(OPEN_METEO_ARCHIVE_URL, params=params, timeout=180)
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 0)) or min(90, 8 * 2 ** attempt)
                print(f"    rate-limit 429 → espero {wait}s (intento {attempt + 1}/{retries})")
                time.sleep(wait)
                continue
            if r.status_code >= 500:
                wait = min(90, 5 * 2 ** attempt)
                print(f"    servidor {r.status_code} → espero {wait}s (intento {attempt + 1}/{retries})")
                time.sleep(wait)
                continue
            r.raise_for_status()
            data = r.json()
            return data if isinstance(data, list) else [data]
        except requests.RequestException as exc:
            wait = min(90, 5 * 2 ** attempt)
            print(f"    error de red/API: {exc}; espero {wait}s (intento {attempt + 1}/{retries})")
            time.sleep(wait)
    raise RuntimeError("Open-Meteo: no se pudo completar el batch tras varios reintentos")


def response_to_daily_rows(chunk: pd.DataFrame, payload: list[dict]) -> list[dict]:
    rows = []
    if len(payload) != len(chunk):
        raise RuntimeError(f"Respuesta inesperada: {len(payload)} ubicaciones para {len(chunk)} solicitadas")

    for (_, req), loc in zip(chunk.iterrows(), payload):
        daily = loc.get("daily", {})
        dates = daily.get("time", [])
        if not dates:
            raise RuntimeError(f"Respuesta sin datos diarios para location_id={req.location_id}")
        for i, date in enumerate(dates):
            row = {
                "location_id": int(req.location_id),
                "source": req.source,
                "commune_name": req.commune_name,
                "lat": float(req.lat),
                "lon": float(req.lon),
                "date": date,
            }
            for var in DAILY_VARS:
                values = daily.get(var, [])
                row[var] = np.nan if i >= len(values) or values[i] is None else values[i]
            rows.append(row)
    return rows


def attach_point_metadata(df: pd.DataFrame, points: gpd.GeoDataFrame) -> pd.DataFrame:
    point_meta = points.drop(columns="geometry")[["location_id", "source", "commune_name", "lat", "lon"]].copy()
    drop = [c for c in ["source", "commune_name", "lat", "lon"] if c in df.columns]
    return df.drop(columns=drop).merge(point_meta, on="location_id", how="left")


def load_or_fetch_daily_climate(points: gpd.GeoDataFrame) -> pd.DataFrame:
    expected_cols = {"location_id", "source", "commune_name", "lat", "lon", "date", *DAILY_VARS}
    if CLIMATE_CACHE.exists():
        df = pd.read_csv(CLIMATE_CACHE)
        df = attach_point_metadata(df, points)
        missing = expected_cols - set(df.columns)
        if missing:
            raise ValueError(f"Caché climática incompleta; faltan columnas: {sorted(missing)}")
        print(f"Clima diario cargado desde caché: {len(df):,} filas")
        return df

    rows = []
    done_ids = set()
    if CLIMATE_PARTIAL.exists():
        partial = pd.read_csv(CLIMATE_PARTIAL)
        rows = attach_point_metadata(partial, points).to_dict("records")
        done_ids = set(partial["location_id"].dropna().astype(int).unique())
        print(f"Reanudando desde parcial: {len(done_ids)} puntos ya descargados")

    points_df = points.drop(columns="geometry").sort_values("location_id").reset_index(drop=True)
    todo = points_df[~points_df["location_id"].isin(done_ids)].copy()
    print(f"Descargando {len(todo)}/{len(points_df)} puntos climáticos desde Open-Meteo...")

    for start in range(0, len(todo), CHUNK_SIZE):
        chunk = todo.iloc[start:start + CHUNK_SIZE].copy()
        payload = fetch_open_meteo_chunk(chunk)
        rows.extend(response_to_daily_rows(chunk, payload))
        pd.DataFrame(rows).to_csv(CLIMATE_PARTIAL, index=False)
        print(f"  {len(set(pd.DataFrame(rows)['location_id'].astype(int)))}/{len(points_df)} puntos")
        time.sleep(REQUEST_SLEEP_S)

    df = pd.DataFrame(rows)
    missing = expected_cols - set(df.columns)
    if missing:
        raise ValueError(f"Descarga climática incompleta; faltan columnas: {sorted(missing)}")
    df.to_csv(CLIMATE_CACHE, index=False)
    if CLIMATE_PARTIAL.exists():
        CLIMATE_PARTIAL.unlink()
    print(f"Guardado en {CLIMATE_CACHE}")
    return df


df_daily = load_or_fetch_daily_climate(gdf_points)
df_daily["date"] = pd.to_datetime(df_daily["date"])
df_daily["month"] = df_daily["date"].dt.month

print(f"Rango temporal: {df_daily['date'].min().date()} → {df_daily['date'].max().date()}")
print(f"Filas: {len(df_daily):,}; puntos: {df_daily['location_id'].nunique()}")
df_daily.head()

## 4. Indicadores por punto climático

Cada punto se resume en métricas epidemiológicamente interpretables: intensidad del calor, frecuencia de días extremos, noches cálidas y precipitación anual.

In [ ]:
def summarize_location(group: pd.DataFrame) -> pd.Series:
    summer = group[group["month"].isin(SUMMER_MONTHS)]
    return pd.Series({
        "tmean_annual_c": group["temperature_2m_mean"].mean(),
        "tmax_mean_annual_c": group["temperature_2m_max"].mean(),
        "summer_tmax_mean_c": summer["temperature_2m_max"].mean(),
        "tmax_p95_c": group["temperature_2m_max"].quantile(0.95),
        "tmax_abs_c": group["temperature_2m_max"].max(),
        "apparent_tmax_mean_c": group["apparent_temperature_max"].mean(),
        "hot_days_30c": (group["temperature_2m_max"] >= HOT_DAY_C).sum(),
        "hot_days_35c": (group["temperature_2m_max"] >= VERY_HOT_DAY_C).sum(),
        "apparent_hot_days_35c": (group["apparent_temperature_max"] >= APPARENT_HOT_DAY_C).sum(),
        "tropical_nights_20c": (group["temperature_2m_min"] >= TROPICAL_NIGHT_C).sum(),
        "precip_annual_mm": group["precipitation_sum"].sum(min_count=1),
        "n_days": group["date"].nunique(),
    })


group_keys = ["location_id", "source", "commune_name", "lat", "lon"]
df_point_metrics = df_daily.groupby(group_keys, dropna=False).apply(summarize_location).reset_index()

gdf_point_metrics = gpd.GeoDataFrame(
    df_point_metrics,
    geometry=gpd.points_from_xy(df_point_metrics["lon"], df_point_metrics["lat"]),
    crs=WGS84,
).to_crs(METRIC_CRS)

metric_cols = [
    "tmean_annual_c", "tmax_mean_annual_c", "summer_tmax_mean_c", "tmax_p95_c", "tmax_abs_c",
    "apparent_tmax_mean_c", "hot_days_30c", "hot_days_35c", "apparent_hot_days_35c",
    "tropical_nights_20c", "precip_annual_mm", "n_days",
]

print(f"Puntos resumidos: {len(gdf_point_metrics)}")
print(gdf_point_metrics[metric_cols].round(2).describe().T[["mean", "std", "min", "max"]])

## 5. Agregación comunal

El indicador principal usa el punto representativo de cada comuna (`source == commune_point`), una aproximación deliberada a exposición residencial cuando no hay direcciones exactas ni grilla poblacional. La grilla regular se usa como diagnóstico de cobertura (`n_area_grid_points`) y como fallback si faltara un punto comunal.

In [ ]:
def zscore(s: pd.Series) -> pd.Series:
    std = s.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(0.0, index=s.index)
    return (s - s.mean()) / std


gdf_comm = gdf_communes[["name", "area_km2", "geometry"]].to_crs(METRIC_CRS).copy()

# Diagnostic area-grid coverage: not the primary exposure estimate.
gdf_grid_metrics = gdf_point_metrics[gdf_point_metrics["source"].eq("grid")].copy()
joined_grid = gpd.sjoin(gdf_grid_metrics, gdf_comm[["name", "geometry"]], how="inner", predicate="within")
area_counts = joined_grid.groupby("name").size().rename("n_area_grid_points").reset_index()

# Primary residential-proxy estimate: one representative point per commune.
rep = pd.DataFrame(gdf_point_metrics[gdf_point_metrics["source"].eq("commune_point")].drop(columns="geometry"))
rep = rep.rename(columns={"commune_name": "name"})
rep = rep.dropna(subset=["name"]).drop_duplicates("name")

gdf_result = gdf_comm.merge(rep[["name", *metric_cols]], on="name", how="left")
gdf_result = gdf_result.merge(area_counts, on="name", how="left")
gdf_result["n_area_grid_points"] = gdf_result["n_area_grid_points"].fillna(0).astype(int)

# Fallback nearest climate point for any commune missing a representative-point estimate.
missing_primary = gdf_result["tmean_annual_c"].isna()
if missing_primary.any():
    cent = gdf_comm.loc[missing_primary, ["name", "geometry"]].copy()
    cent["geometry"] = cent.geometry.representative_point()
    nearest = gpd.sjoin_nearest(
        cent,
        gdf_point_metrics[[*metric_cols, "geometry"]],
        how="left",
        distance_col="nearest_climate_m",
    ).drop(columns="index_right")
    for _, row in nearest.iterrows():
        idx = gdf_result.index[gdf_result["name"].eq(row["name"])][0]
        for col in metric_cols:
            gdf_result.loc[idx, col] = row[col]
        gdf_result.loc[idx, "nearest_climate_m"] = row["nearest_climate_m"]
else:
    gdf_result["nearest_climate_m"] = 0.0

gdf_result["used_nearest_fallback"] = missing_primary.to_numpy()

heat_components = ["summer_tmax_mean_c", "tmax_p95_c", "hot_days_30c", "apparent_hot_days_35c", "tropical_nights_20c"]
for col in heat_components:
    gdf_result[col + "_z"] = zscore(gdf_result[col])
gdf_result["heat_exposure_index"] = gdf_result[[c + "_z" for c in heat_components]].mean(axis=1)
gdf_result["urban_heat_anomaly_c"] = gdf_result["summer_tmax_mean_c"] - gdf_result["summer_tmax_mean_c"].median()

# Tidy rounding for human-readable exports.
round_cols = [c for c in metric_cols if c != "n_days"] + ["nearest_climate_m", "heat_exposure_index", "urban_heat_anomaly_c"]
for col in round_cols:
    if col in gdf_result.columns:
        gdf_result[col] = gdf_result[col].round(2)
gdf_result["area_km2"] = gdf_result["area_km2"].round(2)

if gdf_result["heat_exposure_index"].isna().any():
    bad = gdf_result.loc[gdf_result["heat_exposure_index"].isna(), "name"].tolist()
    raise ValueError(f"Comunas sin índice de calor: {bad}")

print(f"Comunas con punto representativo: {(~gdf_result['used_nearest_fallback']).sum()}/{len(gdf_result)}")
print(f"Comunas con fallback nearest: {gdf_result['used_nearest_fallback'].sum()}")
print(f"Mediana de puntos de grilla por comuna (diagnóstico): {gdf_result['n_area_grid_points'].median():.0f}")
print("\nTop 10 comunas con mayor índice de calor:")
print(gdf_result.sort_values("heat_exposure_index", ascending=False)[
    ["name", "heat_exposure_index", "summer_tmax_mean_c", "hot_days_30c", "hot_days_35c", "tmax_p95_c", "n_area_grid_points"]
].head(10).to_string(index=False))

## 6. Mapa pub-quality — Índice de exposición a calor

Mapa estático centrado en el Gran Santiago para mostrar el gradiente térmico urbano. El índice es relativo dentro de la Región Metropolitana; por diseño sirve para ranking y cruce con otros factores del exposoma.

In [ ]:
LABEL_ALL = [
    "Providencia", "Santiago", "Ñuñoa", "Las Condes", "Vitacura",
    "Maipú", "Puente Alto", "La Florida", "San Bernardo",
    "Independencia", "Cerro Navia", "San Ramón", "Lo Prado",
    "Recoleta", "Conchalí", "Huechuraba", "Quilicura",
    "Peñalolén", "Macul", "San Miguel", "La Cisterna",
    "El Bosque", "La Pintana", "Pedro Aguirre Cerda",
    "Lo Espejo", "Renca", "Cerrillos", "Pudahuel",
    "Estación Central", "Quinta Normal", "San Joaquín", "La Reina", "La Granja",
]

gdf_display = gdf_result[gdf_result["name"].isin(LABEL_ALL)].copy().to_crs(METRIC_CRS)
minx, miny, maxx, maxy = gdf_display.total_bounds
buf = 4_000
X0, X1 = minx - buf, maxx + buf
Y0, Y1 = miny - buf, maxy + buf

vmin = gdf_display["heat_exposure_index"].quantile(0.03)
vmax = gdf_display["heat_exposure_index"].quantile(0.97)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.colormaps["YlOrRd"]

plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.linewidth": 0.8,
})

fig, ax = plt.subplots(figsize=(9, 9))
fig.patch.set_alpha(0)
ax.set_facecolor("none")

gdf_display.plot(
    column="heat_exposure_index", ax=ax, cmap=cmap, norm=norm,
    edgecolor="black", linewidth=1.1,
    missing_kwds={"color": "#dddddd"},
)

ax.set_xlim(X0, X1)
ax.set_ylim(Y0, Y1)

q70 = gdf_display["heat_exposure_index"].quantile(0.70)
for _, row in gdf_display.iterrows():
    c = row.geometry.centroid
    ax.text(
        c.x, c.y, row["name"], fontsize=6.3, ha="center", va="center", color="#111111",
        fontweight="bold" if row["heat_exposure_index"] > q70 else "normal",
        bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.55),
    )

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, aspect=28)
cbar.set_label("Índice de exposición a calor (z compuesto)", fontsize=13, labelpad=8)
cbar.outline.set_linewidth(0.8)
cbar.ax.tick_params(labelsize=11, width=0.8)

dx, dy = X1 - X0, Y1 - Y0
ax.annotate("N", xy=(X1 - dx*0.05, Y0 + dy*0.09), xytext=(X1 - dx*0.05, Y0 + dy*0.05),
            fontsize=11, ha="center", fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.4))
sb_len = 10_000
sb_x, sb_y = X0 + dx*0.04, Y0 + dy*0.025
ax.plot([sb_x, sb_x + sb_len], [sb_y, sb_y], color="black", lw=2.5, solid_capstyle="butt")
ax.text(sb_x + sb_len/2, sb_y + dy*0.008, "10 km", ha="center", va="bottom", fontsize=10)

ax.text(0.99, 0.01, f"Fuente: Open-Meteo Historical Weather API, {YEAR}",
        transform=ax.transAxes, fontsize=8, ha="right", va="bottom",
        color="#555555", fontstyle="italic")
ax.set_axis_off()
plt.tight_layout(pad=0.5)
plt.savefig(OUT_MAP, dpi=300, bbox_inches="tight", transparent=True)
plt.show()
print(f"Mapa guardado: {OUT_MAP}")

## 7. Mapa interactivo

Choropleth comunal con tooltips para inspeccionar las métricas de calor que alimentan el índice.

In [ ]:
gdf_wgs = gdf_result.to_crs(WGS84).copy()

if folium is not None:
    center_geom = gdf_wgs.geometry.union_all().centroid
    center = [center_geom.y, center_geom.x]

    m = folium.Map(location=center, zoom_start=9, tiles="cartodbpositron")

    folium.Choropleth(
        geo_data=gdf_wgs.to_json(),
        data=gdf_wgs,
        columns=["name", "heat_exposure_index"],
        key_on="feature.properties.name",
        fill_color="YlOrRd",
        fill_opacity=0.78,
        line_opacity=0.55,
        nan_fill_color="#dddddd",
        legend_name="Índice de exposición a calor (z compuesto)",
    ).add_to(m)

    folium.GeoJson(
        gdf_wgs,
        name="Comunas",
        style_function=lambda feature: {"fillOpacity": 0, "color": "#333333", "weight": 0.8},
        tooltip=folium.GeoJsonTooltip(
            fields=["name", "heat_exposure_index", "summer_tmax_mean_c", "hot_days_30c", "hot_days_35c", "tmax_p95_c", "precip_annual_mm"],
            aliases=["Comuna", "Índice calor", "Tmax verano (°C)", "Días ≥30°C", "Días ≥35°C", "P95 Tmax (°C)", "Precip. anual (mm)"],
            localize=True,
            sticky=False,
        ),
    ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(OUT_HTML)
    print(f"Mapa interactivo guardado con folium: {OUT_HTML}")
    m
else:
    # Lightweight Leaflet fallback with embedded GeoJSON. It keeps the workflow complete
    # in environments where folium is not installed.
    geojson = gdf_wgs.to_json()
    min_val = float(gdf_wgs["heat_exposure_index"].min())
    max_val = float(gdf_wgs["heat_exposure_index"].max())
    bounds = gdf_wgs.total_bounds.tolist()  # [minx, miny, maxx, maxy]
    html = f'''<!doctype html>
<html lang="es">
<head>
  <meta charset="utf-8" />
  <title>Índice de exposición a calor - RM Santiago</title>
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
  <style>
    html, body, #map {{ height: 100%; margin: 0; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }}
    .legend {{ background: white; padding: 10px 12px; line-height: 1.35; box-shadow: 0 1px 6px rgba(0,0,0,.25); border-radius: 4px; }}
    .legend span {{ display: inline-block; width: 14px; height: 14px; margin-right: 6px; vertical-align: -2px; }}
  </style>
</head>
<body>
<div id="map"></div>
<script>
const geojson = {geojson};
const minVal = {min_val:.6f};
const maxVal = {max_val:.6f};
const bounds = [[{bounds[1]:.6f}, {bounds[0]:.6f}], [{bounds[3]:.6f}, {bounds[2]:.6f}]];
const colors = ["#ffffcc", "#ffeda0", "#fed976", "#feb24c", "#fd8d3c", "#fc4e2a", "#e31a1c", "#bd0026", "#800026"];
function colorFor(v) {{
  if (v === null || Number.isNaN(v)) return "#dddddd";
  const t = Math.max(0, Math.min(0.999, (v - minVal) / (maxVal - minVal || 1)));
  return colors[Math.floor(t * colors.length)];
}}
function style(feature) {{
  const v = feature.properties.heat_exposure_index;
  return {{ fillColor: colorFor(v), weight: 0.8, opacity: 0.75, color: "#333", fillOpacity: 0.78 }};
}}
function tooltip(feature, layer) {{
  const p = feature.properties;
  layer.bindTooltip(
    `<b>${{p.name}}</b><br>` +
    `Índice calor: ${{Number(p.heat_exposure_index).toFixed(2)}}<br>` +
    `Tmax verano: ${{Number(p.summer_tmax_mean_c).toFixed(1)}} °C<br>` +
    `Días ≥30°C: ${{Number(p.hot_days_30c).toFixed(0)}}<br>` +
    `Días ≥35°C: ${{Number(p.hot_days_35c).toFixed(0)}}<br>` +
    `P95 Tmax: ${{Number(p.tmax_p95_c).toFixed(1)}} °C`,
    {{ sticky: false }}
  );
}}
const map = L.map("map");
L.tileLayer("https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png", {{
  attribution: '&copy; OpenStreetMap contributors &copy; CARTO'
}}).addTo(map);
L.geoJSON(geojson, {{ style, onEachFeature: tooltip }}).addTo(map);
map.fitBounds(bounds);
const legend = L.control({{ position: "bottomright" }});
legend.onAdd = function() {{
  const div = L.DomUtil.create("div", "legend");
  div.innerHTML = `<b>Índice de calor</b><br><span style="background:${{colors[0]}}"></span>Menor<br><span style="background:${{colors[8]}}"></span>Mayor`;
  return div;
}};
legend.addTo(map);
</script>
</body>
</html>'''
    OUT_HTML.write_text(html, encoding="utf-8")
    print(f"Mapa interactivo guardado con fallback Leaflet: {OUT_HTML}")

## 8. Ranking comunal y brecha de exposición

El ranking permite comunicar la brecha entre comunas de manera directa para una entrevista o anexo de postulación.

In [ ]:
df_ranked = (
    gdf_result[["name", "heat_exposure_index", "summer_tmax_mean_c", "hot_days_30c", "hot_days_35c", "tmax_p95_c"]]
    .sort_values("heat_exposure_index", ascending=False)
    .reset_index(drop=True)
)
df_ranked.index += 1

fig, ax = plt.subplots(figsize=(10, max(6, len(df_ranked) * 0.34)))
colors = plt.cm.RdYlGn(np.linspace(0, 1, len(df_ranked)))  # rojo = mayor exposición térmica
ax.barh(df_ranked["name"], df_ranked["heat_exposure_index"], color=colors,
        edgecolor="white", linewidth=0.5)
ax.set_xlabel("Índice de exposición a calor (z compuesto)", fontsize=11)
ax.set_title(f"Ranking comunal: exposición a calor ({YEAR})\nRegión Metropolitana de Santiago",
             fontsize=12, fontweight="bold")
ax.invert_yaxis()
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig(OUT_RANKING, dpi=150, bbox_inches="tight")
plt.show()

top, bot = df_ranked.iloc[0], df_ranked.iloc[-1]
print("\n--- Brecha de exposición a calor ---")
print(f"  Mayor exposición: {top['name']} → índice {top['heat_exposure_index']:.2f}; "
      f"Tmax verano {top['summer_tmax_mean_c']:.1f} °C; días ≥35°C: {top['hot_days_35c']:.0f}")
print(f"  Menor exposición: {bot['name']} → índice {bot['heat_exposure_index']:.2f}; "
      f"Tmax verano {bot['summer_tmax_mean_c']:.1f} °C; días ≥35°C: {bot['hot_days_35c']:.0f}")
print(f"  Diferencia Tmax verano: {top['summer_tmax_mean_c'] - bot['summer_tmax_mean_c']:.1f} °C")

## 9. Exportar indicadores

El CSV queda listo para cruzarse con datos clínicos/cognitivos. El GeoJSON conserva la geometría para QGIS, GeoPandas o integración con otros pipelines.

In [ ]:
export_cols = [
    "name", "area_km2",
    "tmean_annual_c", "tmax_mean_annual_c", "summer_tmax_mean_c", "tmax_p95_c", "tmax_abs_c",
    "apparent_tmax_mean_c", "hot_days_30c", "hot_days_35c", "apparent_hot_days_35c",
    "tropical_nights_20c", "precip_annual_mm", "heat_exposure_index", "urban_heat_anomaly_c",
    "n_area_grid_points", "nearest_climate_m", "used_nearest_fallback", "n_days",
]

df_export = pd.DataFrame(gdf_result.drop(columns="geometry"))[export_cols].copy()
df_export.to_csv(OUT_CSV, index=False)
gdf_export_wgs = gdf_result.to_crs(WGS84).copy()
gdf_export_wgs[export_cols + ["geometry"]].to_file(OUT_GEOJSON, driver="GeoJSON")

metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "region": REGION,
    "year": YEAR,
    "source": "Open-Meteo Historical Weather API",
    "api_url": OPEN_METEO_ARCHIVE_URL,
    "model": MODEL or "open-meteo-default-best-match",
    "daily_variables": DAILY_VARS,
    "grid_step_deg": GRID_STEP_DEG,
    "aggregation_method": "commune_representative_point_primary; regular_grid_for_coverage_diagnostic_and_fallback",
    "thresholds_c": {
        "hot_day": HOT_DAY_C,
        "very_hot_day": VERY_HOT_DAY_C,
        "apparent_hot_day": APPARENT_HOT_DAY_C,
        "tropical_night": TROPICAL_NIGHT_C,
    },
    "heat_exposure_components": heat_components,
    "n_communes": int(len(gdf_result)),
    "n_climate_points": int(len(gdf_points)),
    "n_daily_rows": int(len(df_daily)),
}
OUT_META.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))

print("Archivos exportados:")
print(f"  {OUT_CSV}     — indicadores climáticos por comuna")
print(f"  {OUT_GEOJSON} — capa geoespacial")
print(f"  {OUT_META}    — decisiones de procesamiento y umbrales")
print(f"\nDataset final: {len(df_export)} comunas × {len(df_export.columns)} variables")
print("Columnas:", list(df_export.columns))
df_export.sort_values("heat_exposure_index", ascending=False).head()

## 10. Integración con el exposoma construido

Este bloque prueba si el calor se acumula con otros determinantes ya calculados: NSE, áreas verdes, calidad del aire y acceso a salud. Si alguno de esos CSV no está presente, se omite con gracia.

In [ ]:
def maybe_merge(base: pd.DataFrame, path: str, cols: list[str]) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        print(f"Omitido: {path} no existe")
        return base
    other = pd.read_csv(p)
    keep = ["name", *[c for c in cols if c in other.columns]]
    missing = set(cols) - set(other.columns)
    if missing:
        print(f"Advertencia: {path} no contiene {sorted(missing)}")
    return base.merge(other[keep], on="name", how="left")


df_integrated = df_export[["name", "heat_exposure_index", "summer_tmax_mean_c", "hot_days_35c"]].copy()
df_integrated = maybe_merge(df_integrated, "socioeconomic_exposome_rm_santiago.csv", ["nse_index", "pobreza_pct", "ingreso"])
df_integrated = maybe_merge(df_integrated, "green_exposome_rm_santiago.csv", ["green_pct"])
df_integrated = maybe_merge(df_integrated, "air_quality_exposome_rm_santiago.csv", ["pm25_mean", "no2_mean"])
df_integrated = maybe_merge(df_integrated, "healthcare_exposome_rm_santiago.csv", ["density_per_km2", "n_hospital"])

corr_vars = [c for c in ["heat_exposure_index", "nse_index", "pobreza_pct", "green_pct", "pm25_mean", "density_per_km2"] if c in df_integrated.columns]
corr = df_integrated[corr_vars].rank().corr().loc["heat_exposure_index"].drop("heat_exposure_index").sort_values()
print("Correlación de Spearman con heat_exposure_index:")
print(corr.round(2).to_string())

if {"nse_index", "green_pct"}.issubset(df_integrated.columns):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

    axes[0].scatter(df_integrated["nse_index"], df_integrated["heat_exposure_index"],
                    s=36, color="#b2182b", alpha=0.8, edgecolor="white", linewidth=0.5)
    axes[0].axhline(0, color="black", linewidth=0.7, alpha=0.6)
    axes[0].axvline(0, color="black", linewidth=0.7, alpha=0.6)
    axes[0].set_xlabel("Índice socioeconómico (NSE)")
    axes[0].set_ylabel("Índice de exposición a calor")
    axes[0].set_title("Calor vs. NSE")

    axes[1].scatter(df_integrated["green_pct"], df_integrated["heat_exposure_index"],
                    s=36, color="#2166ac", alpha=0.8, edgecolor="white", linewidth=0.5)
    axes[1].axhline(0, color="black", linewidth=0.7, alpha=0.6)
    axes[1].set_xlabel("Área verde accesible (% comuna)")
    axes[1].set_ylabel("Índice de exposición a calor")
    axes[1].set_title("Calor vs. áreas verdes")

    for ax in axes:
        ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUT_INTEGRATION, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figura de integración guardada: {OUT_INTEGRATION}")

# comunas con acumulación potencial: calor alto + NSE bajo o poco verde
risk_cols = ["name", "heat_exposure_index", "summer_tmax_mean_c"]
if "nse_index" in df_integrated.columns:
    df_integrated["low_nse_flag"] = df_integrated["nse_index"] <= df_integrated["nse_index"].quantile(0.25)
    risk_cols.append("nse_index")
if "green_pct" in df_integrated.columns:
    df_integrated["low_green_flag"] = df_integrated["green_pct"] <= df_integrated["green_pct"].quantile(0.25)
    risk_cols.append("green_pct")

print("\nComunas con mayor calor y posible vulnerabilidad acumulada:")
print(df_integrated.sort_values("heat_exposure_index", ascending=False)[risk_cols].head(12).round(2).to_string(index=False))

## 11. Apéndice opcional — Land Surface Temperature vía Google Earth Engine

Este bloque contrasta la temperatura del aire de Open-Meteo con **temperatura superficial terrestre** de MODIS. Es opcional porque requiere cuenta de Earth Engine y proyecto GCP; si no está autenticado, se omite sin romper el notebook.

In [ ]:
GEE_PROJECT = "ee-bayalainostroza"  # ajustar si se usa otro proyecto Earth Engine

try:
    import ee
    ee.Initialize(project=GEE_PROJECT)
    print(f"GEE inicializado → {GEE_PROJECT}")

    communes_4326 = gdf_communes.to_crs(WGS84)
    roi = ee.Geometry.BBox(*communes_4326.total_bounds)
    modis = (
        ee.ImageCollection("MODIS/061/MOD11A2")
        .filterDate(YEAR_START, YEAR_END)
        .filterBounds(roi)
        .select("LST_Day_1km")
    )
    # MOD11A2 scale factor: 0.02 K. Convert to Celsius.
    lst_img = modis.mean().multiply(0.02).subtract(273.15).rename("lst_day_c").clip(roi)

    features = []
    for _, row in communes_4326.iterrows():
        geom_json = json.loads(row.geometry.to_json())
        features.append(ee.Feature(ee.Geometry(geom_json), {"name": row["name"]}))

    reduced = lst_img.reduceRegions(
        collection=ee.FeatureCollection(features),
        reducer=ee.Reducer.mean(),
        scale=1000,
    ).getInfo()

    df_lst = pd.DataFrame([
        {"name": f["properties"]["name"], "modis_lst_day_c": f["properties"].get("mean")}
        for f in reduced["features"]
    ])
    df_lst["modis_lst_day_c"] = df_lst["modis_lst_day_c"].round(2)
    print(f"LST MODIS disponible para {df_lst['modis_lst_day_c'].notna().sum()} comunas")
    print(df_lst.sort_values("modis_lst_day_c", ascending=False).head(10).to_string(index=False))
except Exception as e:
    print("Sección LST omitida (GEE no disponible / no autenticado).")
    print("Para habilitarla: crea un proyecto Earth Engine, ejecuta `earthengine authenticate`,")
    print("ajusta GEE_PROJECT y reejecuta esta celda.")
    print("Detalle:", type(e).__name__, "-", str(e)[:180])

## Cierre

Este notebook convierte la brecha detectada en la oferta —**variables climáticas**— en una salida reproducible y lista para integrar con datos de salud cerebral. La capa resultante se puede usar como covariable ambiental, factor de interacción con NSE o componente de un índice multi-exposoma.

**Siguientes mejoras razonables:**
- Repetir el indicador para múltiples años y construir exposición histórica promedio.
- Sensibilizar umbrales de calor (30/35 °C) y ventanas estacionales.
- Usar red vial o población como ponderadores intra-comuna cuando existan datos residenciales o censales más finos.
- Activar el apéndice MODIS/GEE para comparar temperatura del aire con temperatura superficial urbana.